# Local Housing Database GUI
Run the cell below to launch a native Windows interface window. You can insert records or filter the database using native `psql.exe` without any external dependencies.

In [3]:
import os
import subprocess
import tkinter as tk
from tkinter import messagebox, scrolledtext

# --- 1. Database Configuration Setup ---
PSQL_PATH = r"C:\Programme\PostgreSQL\18\bin\psql.exe"
DB_NAME = "housing"
os.environ["PGPASSWORD"] = "fh-group3-admin"
cmd_base = [PSQL_PATH, "-h", "localhost", "-p", "5432", "-U", "devi-group3", "-d", DB_NAME, "-c"]

def execute_sql(query):
    try:
        res = subprocess.run(cmd_base + [query], capture_output=True, text=True, check=True)
        return res.stdout.strip(), True
    except Exception as e:
        err = getattr(e, "stderr", str(e))
        return err.strip(), False

# --- 2. Action Handlers ---
def save_data():
    sql = f"""INSERT INTO housing_table (median_house_value, income, house_age, rooms) 
              VALUES ({v_val.get()}, {v_inc.get()}, {v_age.get()}, {v_rms.get()}) RETURNING id;"""
    out, success = execute_sql(sql)
    if success:
        messagebox.showinfo("Success", f"Data saved successfully!\n{out.splitlines()[-1]}")
        for entry in [e_val, e_inc, e_age, e_rms]: entry.delete(0, tk.END)
    else:
        messagebox.showerror("Error", f"Failed to save:\n{out}")

def search_data():
    age_from_text = v_fage_from.get().strip()
    age_to_text = v_fage_to.get().strip()
    try:
        age_from = int(age_from_text) if age_from_text else None
        age_to = int(age_to_text) if age_to_text else None
    except ValueError:
        messagebox.showerror("Invalid age", "Enter whole numbers for the age range.")
        return

    if age_from is not None and age_to is not None and age_from > age_to:
        messagebox.showerror("Invalid age range", "The 'from' age must be less than or equal to the 'to' age.")
        return

    sql = "SELECT id, house_age, rooms, income, median_house_value FROM housing_table WHERE 1=1"
    if age_from is not None: sql += f" AND house_age >= {age_from}"
    if age_to is not None: sql += f" AND house_age <= {age_to}"
    sql += " ORDER BY id DESC LIMIT 200;"
    
    out, success = execute_sql(sql)
    txt_output.delete("1.0", tk.END)
    txt_output.insert(tk.END, out if success else f"Search failed:\n{out}")

# --- 3. UI Window Layout Construction ---
root = tk.Tk()
root.title("PostgreSQL Housing Dashboard")
root.geometry("520x580")

# Variables Setup
v_val, v_inc, v_age, v_rms = tk.StringVar(), tk.StringVar(), tk.StringVar(), tk.StringVar()
v_fage_from, v_fage_to = tk.StringVar(), tk.StringVar()

# Part A: Insert Section Frame
f_ins = tk.LabelFrame(root, text=" Insert New Housing Record ", padx=10, pady=10)
f_ins.pack(fill="x", padx=10, pady=5)

labels = ["House Value:", "Income:", "House Age:", "Rooms Counts:"]
vars_list = [v_val, v_inc, v_age, v_rms]
entries = []

for i, label_text in enumerate(labels):
    tk.Label(f_ins, text=label_text).grid(row=i, column=0, sticky="w", pady=2)
    e = tk.Entry(f_ins, textvariable=vars_list[i], width=25)
    e.grid(row=i, column=1, pady=2, padx=10)
    entries.append(e)
e_val, e_inc, e_age, e_rms = entries

tk.Button(f_ins, text="💾 Save Record to DB", command=save_data, bg="#4CAF50", fg="white").grid(row=4, column=0, columnspan=2, pady=8, sticky="we")

# Part B: Search Section Frame
f_src = tk.LabelFrame(root, text=" Filter by House Age (Years) ", padx=10, pady=10)
f_src.pack(fill="x", padx=10, pady=5)

tk.Label(f_src, text="Age from (years):").grid(row=0, column=0, sticky="w")
tk.Entry(f_src, textvariable=v_fage_from, width=10).grid(row=0, column=1, padx=5, pady=2)
tk.Label(f_src, text="Age to (years):").grid(row=0, column=2, sticky="w", padx=10)
tk.Entry(f_src, textvariable=v_fage_to, width=10).grid(row=0, column=3, padx=5, pady=2)

tk.Button(f_src, text="🔍 Select / Filter", command=search_data, bg="#2196F3", fg="white").grid(row=1, column=0, columnspan=4, pady=8, sticky="we")

# Part C: Output Console View Panel
tk.Label(root, text="Database Output (psql format):").pack(anchor="w", padx=10)
txt_output = scrolledtext.ScrolledText(root, height=12, font=("Courier", 9))
txt_output.pack(fill="both", expand=True, padx=10, pady=5)

# Verify Environment Context Layout State before Loop Execution
if not os.path.exists(PSQL_PATH):
    txt_output.insert(tk.END, f"ERROR: psql.exe was not found at:\n{PSQL_PATH}\nPlease check your path settings!")
else:
    search_data()  # Pre-load initial datasets grid layout view

root.mainloop()
